In [98]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os 
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.model_selection import KFold

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV, Ridge

import torch
import torch.nn as nn
import torch.optim as optim

import optuna
from optuna.pruners import SuccessiveHalvingPruner
from optuna.samplers import TPESampler




In [31]:
X_train = pd.read_csv('data/X_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('data/X_test.csv',index_col='ROW_ID')

y_train = pd.read_csv('data/y_train.csv',index_col='ROW_ID')
sample_submission = pd.read_csv('data/sample_submission.csv',index_col='ROW_ID')

# Features

### Basic benchmark features

In [32]:
ret_cols = [c for c in X_test.columns if c.startswith("RET_")]
vol_cols = [c for c in X_test.columns if c.startswith("SIGNED_VOLUME_")]

def fillna_row_mean(df, cols):
    A = df[cols].to_numpy(dtype=float)
    m = np.isnan(A)
    if m.any():
        row_mean = np.nanmean(A, axis=1)
        row_mean = np.where(np.isfinite(row_mean), row_mean, 0.0)
        r, c = np.where(m)
        A[r, c] = row_mean[r]
        df.loc[:, cols] = A
    return df

X_test = fillna_row_mean(X_test, ret_cols)
X_test = fillna_row_mean(X_test, vol_cols)

In [33]:
RET_features = [f'RET_{i}' for i in range(1,20)]
SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1,20)]
TURNOVER_features = ['AVG_DAILY_TURNOVER']

In [34]:
for i in [3,5,10,15,20]:
    X_train[ f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i+1]].mean(1)
    X_train[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_train.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')
    
    X_test[ f'AVERAGE_PERF_{i}'] = X_test[RET_features[:i+1]].mean(1)
    X_test[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_test.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')

In [35]:
features = RET_features + SIGNED_VOLUME_features + TURNOVER_features
features = features + [ f'AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]
features = features + [ f'ALLOCATIONS_AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]

### EMA

In [36]:
ret_cols = [f"RET_{i}" for i in range(1, 21)]
r = X_train[ret_cols].to_numpy()

rtest = X_test[ret_cols].to_numpy()

def ema(arr, L):
    alpha = 2/(L+1)
    w = (1-alpha) ** np.arange(L)  # 0..L-1
    w = w / w.sum()
    return (arr[:, :L] * w).sum(axis=1)

X_train["ema3"]  = ema(r, 3)
X_train["ema5"]  = ema(r, 5)
X_train["ema10"] = ema(r,10)


X_test["ema3"]  = ema(rtest, 3)
X_test["ema5"]  = ema(rtest, 5)
X_test["ema10"] = ema(rtest,10)

m20 = r.mean(axis=1)
s20 = r.std(axis=1, ddof=0)

m20test = rtest.mean(axis=1)
s20test = rtest.std(axis=1, ddof=0)

X_train["z20"] = m20 / (s20 + 1e-12)
X_test["z20"] = m20test / (s20test + 1e-12)

### Streak de signe et entropie 

In [37]:
sign = np.sign(r)  
signtest = np.sign(rtest)

def last_streak_len(sig_row, positive=True):
    # part de RET_1 vers RET_20
    target = 1 if positive else -1
    cnt = 0
    for v in sig_row[:20]: 
        if v == target:
            cnt += 1
        else:
            break
    return cnt

X_train["streak_pos"] = [last_streak_len(s, True) for s in sign]
X_train["streak_neg"] = [last_streak_len(s, False) for s in sign]

X_test["streak_pos"] = [last_streak_len(s, True) for s in signtest]
X_test["streak_neg"] = [last_streak_len(s, False) for s in signtest]

p = (r > 0).mean(axis=1)
p = np.clip(p, 1e-9, 1 - 1e-9)
X_train["sign_entropy20"] = -(p*np.log(p) + (1-p)*np.log(1-p))

ptest = (rtest > 0).mean(axis=1)    
ptest = np.clip(ptest, 1e-9, 1 - 1e-9)
X_test["sign_entropy20"] = -(ptest*np.log(ptest) + (1-ptest)*np.log(1-ptest))


### Qualité de rendement (Sharpe10, Sortino10, t-stat(mean10))

In [38]:
r10 = r[:, :10]
m10 = r10.mean(axis=1)
s10 = r10.std(axis=1, ddof=0)
neg10 = np.minimum(r10, 0)

r10test = rtest[:, :10]
m10test = r10test.mean(axis=1)
s10test = r10test.std(axis=1, ddof=0)
neg10test = np.minimum(r10test, 0)

X_train["sharpe10"]   = m10 / (s10 + 1e-12)
downside10 = np.sqrt((neg10**2).mean(axis=1))
X_train["tstat_mean10"] = m10 / (s10/np.sqrt(10) + 1e-12)

X_test["sharpe10"]   = m10test / (s10test + 1e-12)
downside10test = np.sqrt((neg10test**2).mean(axis=1))
X_test["tstat_mean10"] = m10test / (s10test/np.sqrt(10) + 1e-12)


### Max drawdown et recovery

In [39]:
X_train["vol10"]         = s10
X_train["downside_dev10"]= downside10

X_test["vol10"]         = s10test
X_test["downside_dev10"]= downside10test

def maxdd_and_recovery(row):
    c = np.cumsum(row)                   # equity curve 20j
    peak = np.maximum.accumulate(c)
    dd = (c - peak)
    maxdd = dd.min()                     # drawdown (négatif)
    # recovery: jours depuis le dernier pic
    last_peak_idx = np.where(c == peak)[0][-1]
    recovery = 20 - 1 - last_peak_idx
    return maxdd, recovery

md_rec = np.apply_along_axis(maxdd_and_recovery, 1, r)
X_train["maxdd20"]   = md_rec[:,0]
X_train["recovery20"]= md_rec[:,1]

md_rec_test = np.apply_along_axis(maxdd_and_recovery, 1, rtest)
X_test["maxdd20"]    = md_rec_test[:,0]
X_test["recovery20"] = md_rec_test[:,1]


### Liquidity / turnover (moyenne/écart-type volumes, autocorr, corr ret–liq)

In [40]:

ret_cols = [f"RET_{i}" for i in range(1, 21)]
vol_cols = [f"SIGNED_VOLUME_{i}" for i in range(1, 21)]
EPS = 1e-12

def add_volume_features(X):
    v = np.nan_to_num(X[vol_cols].to_numpy(), nan=0.0)
    r = np.nan_to_num(X[ret_cols].to_numpy(), nan=0.0)

    X["sv_mean20"] = v.mean(axis=1)
    X["sv_std20"]  = v.std(axis=1, ddof=0)

    v_mean = v.mean(axis=1, keepdims=True)
    vx = v - v_mean
    num = (vx[:,1:] * vx[:,:-1]).sum(axis=1)
    den = np.sqrt((vx[:,1:]**2).sum(axis=1) * (vx[:,:-1]**2).sum(axis=1))
    X["sv_autocorr1"] = num / (den + EPS)

    rx = r - r.mean(axis=1, keepdims=True)
    num = (rx * vx).sum(axis=1)
    den = np.sqrt((rx**2).sum(axis=1) * (vx**2).sum(axis=1))
    X["corr_ret_sv20"] = num / (den + EPS)

    return X

X_train = add_volume_features(X_train.copy())
X_test  = add_volume_features(X_test.copy())


In [41]:
features = [col for col in X_train.columns if col not in ['TS', 'ALLOCATION']]

# Preprocessing

### Winsorizer et SVD

In [ ]:

class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.005, upper=0.995):
        self.lower = lower; self.upper = upper
    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.lo_ = X.quantile(self.lower)
        self.hi_ = X.quantile(self.upper)
        return self
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        return X.clip(self.lo_, self.hi_, axis=1).values

# ---- group SVD encoder (fit on train groups, transform new data) ----
class GroupSVDEncoder:
    """
    Fit: on train_df -> groupby(group_col)[feature_cols].agg('mean') -> scale -> SVD
    Transform: takes any df, computes group means in *that* df, scales with train scaler, projects with train SVD,
               then merges per-row by group id.
    """
    def __init__(self, group_col, feature_cols, var_tol=1e-5, n_components=8, random_state=42, agg='mean', suffix=''):
        self.group_col = group_col
        self.feature_cols = feature_cols
        self.n_components = n_components
        self.random_state = random_state
        self.agg = agg

        self.var_tol = var_tol
        
        self.suffix = suffix or group_col.lower()

    def fit(self, df_train):
        g = getattr(df_train[self.feature_cols + [self.group_col]].groupby(self.group_col), self.agg)()
    
        std = g.std(axis=0, ddof=0)
        self.keep = std[std > self.var_tol ].index
        dropped = sorted(set(g.columns) - set(self.keep))
        self.dropped_cols_ = dropped
        g = g[self.keep]
    
        self.scaler_ = StandardScaler().fit(g.values)
        G = self.scaler_.transform(g.values)

        self.svd_ = TruncatedSVD(n_components=self.n_components, svd_solver="full", random_state=self.random_state)
        Z = self.svd_.fit_transform(G)

        self.cols_ = [f"{self.suffix}_svd_{i+1}" for i in range(self.n_components)]
        self.train_group_emb_ = pd.DataFrame(Z, index=g.index, columns=self.cols_)
        self.ev_ = float(getattr(self.svd_, "explained_variance_ratio_", [0]).sum())

        return self

    def transform(self, df_any):
        # build embeddings for groups present in df_any (can be unseen groups)
        g_any = getattr(df_any[self.feature_cols + [self.group_col]].groupby(self.group_col), self.agg)()

        g_any = g_any[self.keep]

        Gt = self.scaler_.transform(g_any.values) 

        Zt = self.svd_.transform(Gt)
        emb = pd.DataFrame(Zt, index=g_any.index, columns=self.cols_)
        emb[self.suffix + "_svd_ratio"] = self.ev_
        # merge back per row (left join on group id)
        return df_any.merge(emb.reset_index(), on=self.group_col, how="left")


### Denoizing AutoEncoder Transformer

In [43]:


use_dae = True

class DAETransformer(BaseEstimator, TransformerMixin):
    """
    Scikit-learn compatible DAE:
    - standardize inputs
    - add Gaussian noise during training
    - return latent code concatenated to original inputs (or just code if return_code_only=True)
    """
    def __init__(self, n_hidden=128, code_dim=32, noise_std=0.05, epochs=10, batch_size=512, lr=1e-3,
                 return_code_only=True, device='auto', random_state=42):
        self.n_hidden = n_hidden
        self.code_dim = code_dim
        self.noise_std = noise_std
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.return_code_only = return_code_only
        self.device = device
        self.random_state = random_state

    def _build(self, in_dim):
        enc = nn.Sequential(
            nn.Linear(in_dim, self.n_hidden), nn.ReLU(),
            nn.Linear(self.n_hidden, self.code_dim)
        )
        dec = nn.Sequential(
            nn.Linear(self.code_dim, self.n_hidden), nn.ReLU(),
            nn.Linear(self.n_hidden, in_dim)
        )
        return enc, dec

    def fit(self, X, y=None):
        rs = np.random.RandomState(self.random_state)
        X = np.asarray(X, dtype=np.float32)
        self.scaler_ = StandardScaler().fit(X)
        Xs = self.scaler_.transform(X).astype(np.float32)
        in_dim = Xs.shape[1]
        self.encoder_, self.decoder_ = self._build(in_dim)
        dev = torch.device('cuda' if (self.device=='auto' and torch.cuda.is_available()) else 'cpu')
        self.encoder_.to(dev); self.decoder_.to(dev)
        opt = optim.Adam(list(self.encoder_.parameters())+list(self.decoder_.parameters()), lr=self.lr)
        loss_fn = nn.MSELoss()

        ds = torch.utils.data.TensorDataset(torch.from_numpy(Xs))
        dl = torch.utils.data.DataLoader(ds, batch_size=self.batch_size, shuffle=True, drop_last=False)

        self.encoder_.train(); self.decoder_.train()
        for _ in range(self.epochs):
            for (xb,) in dl:
                xb = xb.to(dev)
                noise = torch.from_numpy(rs.normal(0, self.noise_std, xb.shape).astype(np.float32)).to(dev)
                xnoisy = xb + noise
                code = self.encoder_(xnoisy)
                recon = self.decoder_(code)
                loss = loss_fn(recon, xb)
                opt.zero_grad(); loss.backward(); opt.step()
        self.encoder_.eval()
        self.device_ = dev
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=np.float32)
        Xs = self.scaler_.transform(X).astype(np.float32)
        with torch.no_grad():
            codes = self.encoder_(torch.from_numpy(Xs).to(self.device_)).cpu().numpy()
        if self.return_code_only:
            return codes
        return np.concatenate([X, codes], axis=1)


# Ridge training - Kfold CV

### Kfold init 

In [105]:
from sklearn.model_selection import GroupKFold

n_splits = 10
#kf = KFold(n_splits=n_splits, shuffle=True, random_state=43)


groups = X_train["TS"]  

train_dates = X_train['TS'].unique()
scores = []
models = []
best_rounds = []
best_scores = []

y_train_bin = y_train.copy()
y_train_bin["target"] = (y_train_bin["target"] > 0).astype(int)


# OOF containers
oof_pred = np.full(len(X_train), np.nan)
oof_cls  = np.zeros(len(X_train), dtype=int)
fold_acc = []
fold_models = []
fold_selectors = []
fold_thresholds = []

test_votes = np.zeros((n_splits, len(X_test)))


gkf = GroupKFold(n_splits=n_splits)


### Feature selection 

Feature selector:
1) Split data into K folds (GroupKFold if groups provided)
2) For each fold, compute per-feature Spearman rho(X_j, y) on the train part
3) Aggregate per-feature |rho| by a robust quantile (stability_q, e.g., median)
4) Sort features by score desc, then greedily drop features highly correlated
         (|Pearson| >= col_thresh) with already-selected ones
5) Keep top max_keep features

In [ ]:


class RobustFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self,
                 groups=None,
                 k_folds=5,
                 col_thresh=0.95,
                 stability_q=0.5,
                 max_keep=100,
                 standardize=True,
                 random_state=42):
        self.groups = groups
        self.k_folds = k_folds
        self.col_thresh = col_thresh
        self.stability_q = stability_q
        self.max_keep = max_keep
        self.standardize = standardize
        self.random_state = random_state

    def _ensure_df(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X).copy()

    def fit(self, X, y):
        X = self._ensure_df(X)
        y = np.asarray(y).ravel()

    
        X = X[features]
        self.feature_names_in_ = features

        n = len(X)
        groups = None
        if self.groups is not None:
            groups = np.asarray(self.groups)
            assert len(groups) == n, "groups length must match X."

        # splitter
        if groups is not None:
            splitter = GroupKFold(n_splits=self.k_folds)
            split_iter = splitter.split(X, y, groups=groups)
        else:
            splitter = KFold(n_splits=self.k_folds, shuffle=True, random_state=self.random_state)
            split_iter = splitter.split(X, y)

        # compute per-fold medians for imputation + per-fold Spearman |rho|
        rhos = [] 
        for tr_idx, _ in split_iter:
            Xtr = X.iloc[tr_idx]
            ytr = y[tr_idx]

            # median impute (fit on train fold)
            med = Xtr.median(axis=0)
            Xtr_imp = Xtr.fillna(med)

            # Spearman |rho|, per feature (simple & readable)
            vals = []
            rank_y = pd.Series(ytr).rank(method="average").to_numpy()
            for c in Xtr_imp.columns:
                x = Xtr_imp[c].to_numpy()
                # rank x (Spearman)
                rank_x = pd.Series(x).rank(method="average").to_numpy()
                # pearson on ranks
                vx = rank_x - rank_x.mean()
                vy = rank_y - rank_y.mean()
                denom = np.sqrt((vx**2).sum() * (vy**2).sum())
                if denom == 0:
                    rho = 0.0
                else:
                    rho = (vx @ vy) / denom
                vals.append(abs(rho))
            rhos.append(np.asarray(vals, dtype=float))

        R = np.vstack(rhos)  # (k_folds, n_features)
        # robust aggregate score per feature
        q = np.clip(self.stability_q, 0.0, 1.0)
        agg = np.quantile(R, q=q, axis=0)  # shape (n_features,)

        # sort features by score desc
        order = np.argsort(-agg)
        sorted_feats = [self.feature_names_in_[i] for i in order]
        sorted_scores = agg[order]

        # prepare dataset for redundancy filtering (global medians + optional standardize)
        med_global = X.median(axis=0)
        X_imp = X.fillna(med_global)
        if self.standardize:
            scaler = StandardScaler(with_mean=True, with_std=True)
            X_std = pd.DataFrame(scaler.fit_transform(X_imp), columns=X_imp.columns, index=X_imp.index)
        else:
            X_std = X_imp

        # greedy redundancy filter
        kept = []
        for f in sorted_feats:
            if len(kept) >= self.max_keep:
                break
            keep = True
            xf = X_std[f]
            for g in kept:
                corr = float(np.corrcoef(xf, X_std[g])[0, 1])
                if np.isnan(corr):
                    corr = 0.0
                if abs(corr) >= self.col_thresh:
                    keep = False
                    break
            if keep:
                kept.append(f)

        self.selected_features_ = kept
        self.scores_ = {feat: float(score) for feat, score in zip(sorted_feats, sorted_scores)}
        self.medians_ = med_global
        self.scaler_ = None
        if self.standardize:
            self.scaler_ = StandardScaler(with_mean=True, with_std=True).fit(X_imp[self.selected_features_])
        return self

    def transform(self, X):
        X = self._ensure_df(X)
        # only keep columns seen during fit; silently drop missing
        cols = [c for c in self.selected_features_ if c in X.columns]
        X = X[cols].copy()

        # impute with training medians (for those columns)
        for c in cols:
            if c not in self.medians_:
                continue
            
            X[c] = X[c].fillna(self.medians_[c])

        # (optional) standardize to the training scaler (kept features only)
        if self.standardize and self.scaler_ is not None:
            X.loc[:, cols] = self.scaler_.transform(X[cols])

        return X

    # convenience helper
    def get_support(self):
        """Boolean mask aligned to feature_names_in_ (True if kept)."""
        keep = set(self.selected_features_)
        return np.array([c in keep for c in self.feature_names_in_], dtype=bool)

    def get_feature_names_out(self):
        return np.array(self.selected_features_, dtype=object)


## Model training

In [ ]:
{'R_ALLOC': 6, 'R_TS': 6, 'win_low': 0.004146300254424145, 'win_up': 0.9971757737599501, 'scaler': 'standard', 'alpha': 63625.03199147733, 'stability_q': 0.4, 'col_thresh': 0.9439592191203193}

In [119]:

i = 1
selected_features = []

# ridge alpha grid (yours)
#ALPHAS = [np.logspace(1, 6, 10)]
ALPHAS = [63625]

r_alloc = 6
r_ts = 6


num_pre = Pipeline([
        ("imp", SimpleImputer(strategy="median", add_indicator=True)),
        ("win", Winsorizer(0.004, 0.997)),
        ("scale", StandardScaler()),
    ])

dae = DAETransformer(n_hidden=128, code_dim=32, noise_std=0.05, epochs=12, batch_size=1024, lr=1e-3,
                     return_code_only=True, random_state=42) if use_dae else None

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups=X_train["TS"])):

    x_tr, y_tr = X_train.iloc[tr_idx].copy(), y_train.iloc[tr_idx].copy()
    x_val, y_val = X_train.iloc[val_idx].copy(), y_train.iloc[val_idx].copy()
    x_test = X_test.copy()

    y_val_bin = (y_val["target"] > 0).astype(int)

    # pipeline de préprocessing 
    num_pre_fit = clone(num_pre).fit(x_tr[features])
    Xtr_num = num_pre_fit.transform(x_tr[features])
    Xval_num = num_pre_fit.transform(x_val[features])
    Xte_num  = num_pre_fit.transform(x_test[features])

    
    # embeddings SVD allocations / dates
    alloc_encoder = GroupSVDEncoder(group_col="ALLOCATION",
                                    feature_cols=features, n_components=r_alloc, suffix="alloc").fit(x_tr)
    x_tr  = alloc_encoder.transform(x_tr)
    x_val = alloc_encoder.transform(x_val)
    x_test = alloc_encoder.transform(x_test)
    alloc_cols = [c for c in x_tr.columns if c.startswith("alloc_svd_")] + ["alloc_svd_ratio"]

    
    ts_encoder = GroupSVDEncoder(group_col="TS",
                                feature_cols=features, n_components=r_ts, suffix="ts").fit(x_tr)
    x_tr  = ts_encoder.transform(x_tr)
    x_val = ts_encoder.transform(x_val)
    x_test= ts_encoder.transform(x_test)

    ts_cols = [c for c in x_tr.columns if c.startswith("ts_svd_")] + ["ts_svd_ratio"]

    Xtr = np.column_stack([Xtr_num,  x_tr[alloc_cols].to_numpy(),  x_tr[ts_cols].to_numpy()])
    Xval= np.column_stack([Xval_num, x_val[alloc_cols].to_numpy(), x_val[ts_cols].to_numpy()])
    Xte = np.column_stack([Xte_num,  x_test[alloc_cols].to_numpy(), x_test[ts_cols].to_numpy()])

    # DAE
    if use_dae:
        dae_fit = clone(dae).fit(Xtr)
        Ztr, Zval, Zte = (dae_fit.transform(Xtr),
                                dae_fit.transform(Xval),
                                dae_fit.transform(Xte))
        Xtr   = np.column_stack([Xtr, Ztr])
        Xval  = np.column_stack([Xval, Zval])
        Xte   = np.column_stack([Xte,  Zte])
        

    feat_names = [f"f_{k}" for k in range(Xtr.shape[1])]
    Xtr_df  = pd.DataFrame(Xtr,  columns=feat_names)
    Xval_df = pd.DataFrame(Xval, columns=feat_names)
    Xte_df  = pd.DataFrame(Xte,  columns=feat_names)

    # sélection de features
    # selector = RobustFeatureSelector(
    #     groups=X_train.loc[tr_idx, "TS"],
    #     k_folds=3, col_thresh=0.94, stability_q=0.4, standardize=True,
    # )
    # selector.fast_spearman = True
    # selector.mi_subsample_rows = 20000

   
    # Xtr_sel  = selector.fit_transform(Xtr_df, np.asarray(y_tr).ravel())
    # Xval_sel = selector.transform(Xval_df)
    # Xte_sel  = selector.transform(Xte_df)
    
    # Régression Ridge
    ridge = RidgeCV(alphas=ALPHAS, cv=5, scoring="neg_mean_squared_error", fit_intercept=False).fit(Xtr, y_tr)

    # recherche de treshold de prédiction 
    p_val = ridge.predict(Xval)

    acc = accuracy_score(y_val_bin, (p_val >= 0).astype(int))
    # cand = np.unique(np.quantile(p_val, np.linspace(0.3, 0.7, 31)))
    # cand = np.append(cand, 0.0)
    # accs = [(thr, accuracy_score(y_val_bin, (p_val >= thr).astype(int))) for thr in cand]
    # t_star, acc_star = max(accs, key=lambda x: x[1])

    oof_pred[val_idx] = p_val
    oof_cls[val_idx]  = (p_val >=0).astype(int)
    fold_acc.append(acc)

    # test votes
    p_tst = ridge.predict(Xte)
    test_votes[i-1, :] = (p_tst >= 0).astype(int)

    # stock
    fold_models.append(ridge)
   

    print(f"Fold {i:02d} | alpha*={ridge.alpha_:.4g} | Acc(val)={acc*100:.2f}%")
    i += 1

oof_acc = accuracy_score((y_train["target"]>0).astype(int), oof_cls)
print(f"OOF Accuracy = {oof_acc*100:.2f}% | mean(fold Acc) = {np.mean(fold_acc)*100:.2f}% ± {np.std(fold_acc)*100:.2f}%")



Fold 01 | alpha*=6.362e+04 | Acc(val)=51.87%
Fold 02 | alpha*=6.362e+04 | Acc(val)=51.31%
Fold 03 | alpha*=6.362e+04 | Acc(val)=52.32%
Fold 04 | alpha*=6.362e+04 | Acc(val)=51.79%
Fold 05 | alpha*=6.362e+04 | Acc(val)=53.15%
Fold 06 | alpha*=6.362e+04 | Acc(val)=52.23%
Fold 07 | alpha*=6.362e+04 | Acc(val)=52.44%
Fold 08 | alpha*=6.362e+04 | Acc(val)=52.80%
Fold 09 | alpha*=6.362e+04 | Acc(val)=51.50%
Fold 10 | alpha*=6.362e+04 | Acc(val)=52.33%
OOF Accuracy = 52.17% | mean(fold Acc) = 52.37% ± 0.59%


### To submission

In [121]:
vote_share = test_votes.mean(axis=0)  

# vote de majorité
pred = (vote_share >= 0.5).astype(int)


sub = pd.DataFrame({
    "ROW_ID": X_test.index,
    "prediction": pred,
})

sub[["ROW_ID", "prediction"]].to_csv("data/clean_sub_votes_no_fs.csv", index=False)
print(sub.head())

   ROW_ID  prediction
0  180245           0
1  180246           1
2  180247           0
3  180248           0
4  180249           0


# Hp search

In [ ]:
N_PARALLEL_TRIALS = 3     
BLAS_THREADS = "1"           

os.environ["OMP_NUM_THREADS"] = BLAS_THREADS
os.environ["MKL_NUM_THREADS"] = BLAS_THREADS
os.environ["OPENBLAS_NUM_THREADS"] = BLAS_THREADS
os.environ["NUMEXPR_NUM_THREADS"] = BLAS_THREADS


y_full = (y_train["target"].to_numpy().ravel()
          if isinstance(y_train, pd.DataFrame) else np.asarray(y_train).ravel())

# -------------------- préproc numérique (tunable) --------------------
def make_num_pre(lower_q, upper_q, scaler_type):
    scaler = RobustScaler(with_centering=True, with_scaling=True) if scaler_type == "robust" \
             else StandardScaler(with_mean=True, with_std=True)
    return Pipeline([
        ("imp", SimpleImputer(strategy="median", add_indicator=True)),
        ("win", Winsorizer(lower_q, upper_q)),
        ("scale", scaler),
    ])

# -------------------- PRÉCOMPUTE: embeddings SVD par fold (k max) --------------------
# on choisit k_max = 12 (max des valeurs candidates)
K_MAX = 12
gkf_outer = GroupKFold(n_splits=5)
precomp = []

for fold_id, (tr_idx, va_idx) in enumerate(gkf_outer.split(X_train, y_full, groups=X_train["TS"]), start=1):
    x_tr = X_train.iloc[tr_idx].copy()
    x_va = X_train.iloc[va_idx].copy()

    # Fit 1x avec k_max
    alloc_enc_max = GroupSVDEncoder(group_col="ALLOCATION", feature_cols=features,
                                    n_components=K_MAX, suffix="alloc").fit(x_tr)
    ts_enc_max    = GroupSVDEncoder(group_col="TS", feature_cols=features,
                                    n_components=K_MAX, suffix="ts").fit(x_tr)

    # Transform 1x train/valid (ordre: alloc -> ts)
    x_tr_e = ts_enc_max.transform(alloc_enc_max.transform(x_tr))
    x_va_e = ts_enc_max.transform(alloc_enc_max.transform(x_va))

    alloc_cols_all = [c for c in x_tr_e.columns if c.startswith("alloc_svd_")]
    ts_cols_all    = [c for c in x_tr_e.columns if c.startswith("ts_svd_")]

    precomp.append({
        "tr_idx": tr_idx, "va_idx": va_idx,
        "x_tr_e": x_tr_e, "x_va_e": x_va_e,
        "alloc_cols_all": alloc_cols_all,
        "ts_cols_all": ts_cols_all
    })

def objective(trial: optuna.Trial):
    params = {
        # dims SVD (on slice les colonnes pré-calculées)
        "R_ALLOC": trial.suggest_categorical("R_ALLOC", [6, 8, 12]),
        "R_TS":    trial.suggest_categorical("R_TS",    [6, 8, 12]),
        # préproc
        "win_low": trial.suggest_float("win_low", 0.002,  0.005),
        "win_up":  trial.suggest_float("win_up",  0.995, 0.998),
        "scaler":  trial.suggest_categorical("scaler", ["robust", "standard"]),
        # ridge sans nested CV
        "alpha":   trial.suggest_float("alpha", 1e+1, 1e+6, log=True),
        
        "stability_q": trial.suggest_categorical("stability_q", [0.3, 0.4, 0.5]),
        "col_thresh":  trial.suggest_float("col_thresh", 0.90, 0.98),
    }

    rmse_list = []
    for step, pc in enumerate(precomp, start=1):
        tr_idx, va_idx = pc["tr_idx"], pc["va_idx"]
        x_tr, y_tr = X_train.iloc[tr_idx], y_full[tr_idx]
        x_va, y_va = X_train.iloc[va_idx], y_full[va_idx]

        # 1) num preprocess (fit sur x_tr)
        num_pre = make_num_pre(params["win_low"], params["win_up"], params["scaler"])
        num_pre_fit = clone(num_pre).fit(x_tr[features])
        Xtr_num = num_pre_fit.transform(x_tr[features])
        Xva_num = num_pre_fit.transform(x_va[features])

        # 2) slice des embeddings pré-calculés
        k_alloc, k_ts = params["R_ALLOC"], params["R_TS"]
        alloc_cols = pc["alloc_cols_all"][:k_alloc] + ["alloc_svd_ratio"]
        ts_cols    = pc["ts_cols_all"][:k_ts]       + ["ts_svd_ratio"]

        x_tr_e, x_va_e = pc["x_tr_e"], pc["x_va_e"]
        Xtr = np.column_stack([Xtr_num, x_tr_e[alloc_cols].to_numpy(), x_tr_e[ts_cols].to_numpy()])
        Xva = np.column_stack([Xva_num, x_va_e[alloc_cols].to_numpy(), x_va_e[ts_cols].to_numpy()])

        # 3) sélection robuste (light pendant la search)
        feat_names = [f"f_{i}" for i in range(Xtr.shape[1])]
        Xtr_df = pd.DataFrame(Xtr, columns=feat_names)
        Xva_df = pd.DataFrame(Xva, columns=feat_names)

        selector = RobustFeatureSelector(
            groups=None, k_folds=2,  # <-- plus léger qu'en final
            col_thresh=params["col_thresh"],
            stability_q=params["stability_q"],
            standardize=True, random_state=42
        )
        Xtr_sel = selector.fit_transform(Xtr_df, y_tr)
        Xva_sel = selector.transform(Xva_df)

        # 4) ridge (alpha direct via Optuna, pas de nested CV)
        ridge = Ridge(alpha=params["alpha"], fit_intercept=True, random_state=42)
        ridge.fit(Xtr_sel, y_tr)
        p = ridge.predict(Xva_sel)
        rmse = float(np.sqrt(mean_squared_error(y_va, p)))
        rmse_list.append(rmse)

        # 5) pruning agressif: rapport dès le 1er fold
        trial.report(np.mean(rmse_list), step=step)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(rmse_list))

# -------------------- ÉTUDE OPTUNA --------------------
sampler = TPESampler(seed=42, n_startup_trials=10, multivariate=True)
pruner  = SuccessiveHalvingPruner(min_resource=1, reduction_factor=2)

study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)


study.optimize(objective, n_trials=150, n_jobs=N_PARALLEL_TRIALS, show_progress_bar=True)

print("Best RMSE:", study.best_value*100)
print("Best params:", study.best_params)
best_params = study.best_params


[I 2025-10-13 17:58:16,219] A new study created in memory with name: no-name-38cce7fb-6346-459a-b10b-4e837c30fbd0


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2025-10-13 17:58:29,385] Trial 2 pruned. 
[I 2025-10-13 17:58:29,471] Trial 1 pruned. 
[I 2025-10-13 17:58:39,862] Trial 4 pruned. 
[I 2025-10-13 17:58:52,679] Trial 3 pruned. 
[I 2025-10-13 17:58:54,513] Trial 5 pruned. 
[I 2025-10-13 17:59:01,262] Trial 6 pruned. 
[I 2025-10-13 17:59:03,449] Trial 7 pruned. 
[I 2025-10-13 17:59:07,310] Trial 0 finished with value: 0.0016044958866128296 and parameters: {'R_ALLOC': 6, 'R_TS': 8, 'win_low': 0.0047660786339539405, 'win_up': 0.9955464999955863, 'scaler': 'standard', 'alpha': 8258.678570891223, 'stability_q': 0.3, 'col_thresh': 0.9504112040108875}. Best is trial 0 with value: 0.0016044958866128296.
[I 2025-10-13 17:59:13,018] Trial 9 pruned. 
[I 2025-10-13 17:59:22,429] Trial 11 pruned. 
[I 2025-10-13 17:59:26,601] Trial 10 pruned. 
[I 2025-10-13 17:59:58,260] Trial 8 finished with value: 0.0016040171283269004 and parameters: {'R_ALLOC': 8, 'R_TS': 8, 'win_low': 0.0038000471963068215, 'win_up': 0.9954544672673366, 'scaler': 'robust', 'a